In [ ]:
"""
Synthetic data generator for the Cross-Border Tax Document Assistant.

Creates fake (100% synthetic) PDFs:
  - invoices           (with planted anomalies: duplicate numbers, missing fields, outlier amounts)
  - bank statements    (foreign accounts; some cross the FBAR $10,000 aggregate threshold)
  - W-8BEN style forms (simplified sample, NOT the official IRS form)
  - 1099-NEC style forms (simplified sample, NOT the official IRS form)

Also writes ground_truth.csv so you can measure extraction accuracy and
check that your anomaly flags catch what was planted.

Setup:  pip install reportlab
Run:    python generate_synthetic_data.py
Output: samples/ folder + samples/ground_truth.csv + samples/clients.csv
"""
import csv
import random
from datetime import date, timedelta
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

random.seed(42)  # reproducible
OUT = Path("samples")
OUT.mkdir(exist_ok=True)
styles = getSampleStyleSheet()

# ---------- fake reference data ----------
FIRST = ["Aarav", "Meera", "Liam", "Sofia", "Kenji", "Amara", "Lucas", "Ananya", "Noah", "Elena"]
LAST = ["Sharma", "Patel", "Tanaka", "Silva", "Muller", "Okafor", "Rossi", "Nair", "Kim", "Dubois"]
VENDORS = ["Northwind Consulting", "Bluepeak Software", "Greenleaf Logistics",
           "Orion Legal Services", "Summit Cloud Ltd", "Harbor Analytics", "Zenith Marketing"]
COUNTRIES = {  # country: (currency, bank name)
    "India": ("INR", "Sample Bank of India"),
    "United Kingdom": ("GBP", "Example Bank UK"),
    "Germany": ("EUR", "Beispiel Bank AG"),
    "Singapore": ("SGD", "Demo Bank Singapore"),
    "Japan": ("JPY", "Sakura Sample Bank"),
}
# rough rates to USD, for FBAR ground truth only
TO_USD = {"INR": 0.012, "GBP": 1.27, "EUR": 1.08, "SGD": 0.74, "JPY": 0.0067, "USD": 1.0}
CURRENCY_SYMBOL = {"INR": "INR ", "GBP": "GBP ", "EUR": "EUR ", "SGD": "SGD ", "JPY": "JPY ", "USD": "USD "}

ground_truth = []


def rand_date(start=date(2025, 1, 1), days=365):
    return start + timedelta(days=random.randint(0, days))


def build(path, story):
    SimpleDocTemplate(str(path), pagesize=letter, topMargin=0.7 * inch,
                      bottomMargin=0.7 * inch).build(story)


def table(data, col_widths=None, header=True):
    t = Table(data, colWidths=col_widths)
    style = [("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
             ("FONTSIZE", (0, 0), (-1, -1), 9),
             ("VALIGN", (0, 0), (-1, -1), "TOP")]
    if header:
        style += [("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
                  ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold")]
    t.setStyle(TableStyle(style))
    return t


# ---------- clients ----------
def make_clients(n=8):
    clients = []
    for i in range(1, n + 1):
        name = f"{random.choice(FIRST)} {random.choice(LAST)}"
        country = random.choice(list(COUNTRIES))
        tin = f"SYN-{random.randint(100000, 999999)}"  # obviously fake ID
        clients.append({"client_id": i, "name": name, "country": country, "tin": tin})
    with open(OUT / "clients.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=clients[0].keys())
        w.writeheader()
        w.writerows(clients)
    return clients


# ---------- invoices ----------
def make_invoice(client, vendor, inv_no, inv_date, amount, currency="USD",
                 missing=None, anomaly="none", filename=None):
    """missing: list of field names to leave out of the document."""
    missing = missing or []
    filename = filename or f"invoice_{inv_no or 'NOINV'}_{client['client_id']}.pdf"
    story = [Paragraph(f"<b>{vendor}</b>", styles["Title"]), Paragraph("INVOICE", styles["Heading2"])]

    meta = []
    if "invoice_number" not in missing:
        meta.append(["Invoice No:", inv_no])
    if "date" not in missing:
        meta.append(["Date:", inv_date.isoformat()])
    meta.append(["Bill To:", client["name"]])
    meta.append(["Country:", client["country"]])
    story += [table(meta, [1.5 * inch, 3 * inch], header=False), Spacer(1, 14)]

    qty = random.randint(1, 10)
    unit = round(amount / qty, 2)
    story.append(table([["Description", "Qty", "Unit Price", "Amount"],
                        ["Professional services", qty, f"{unit:,.2f}", f"{unit * qty:,.2f}"]],
                       [3 * inch, 0.7 * inch, 1.2 * inch, 1.2 * inch]))
    story.append(Spacer(1, 10))
    if "total_amount" not in missing:
        story.append(Paragraph(f"<b>Total ({currency}): {amount:,.2f}</b>", styles["Normal"]))
    story.append(Spacer(1, 8))
    story.append(Paragraph("SYNTHETIC SAMPLE DOCUMENT - NOT REAL", styles["Italic"]))
    build(OUT / filename, story)

    ground_truth.append({
        "file": filename, "doc_type": "invoice", "client_id": client["client_id"],
        "vendor": vendor, "invoice_number": inv_no if "invoice_number" not in missing else "",
        "date": inv_date.isoformat() if "date" not in missing else "",
        "currency": currency, "total_amount": amount if "total_amount" not in missing else "",
        "anomaly": anomaly, "note": ",".join(missing),
    })


def generate_invoices(clients, n_normal=14):
    # each vendor has a typical amount so "unusual amount" is meaningful
    typical = {v: random.randint(800, 4000) for v in VENDORS}
    used_numbers = []
    for i in range(n_normal):
        c, v = random.choice(clients), random.choice(VENDORS)
        inv_no = f"INV-{2025}-{1000 + i}"
        amt = round(typical[v] * random.uniform(0.85, 1.15), 2)
        make_invoice(c, v, inv_no, rand_date(), amt)
        used_numbers.append((c, v, inv_no, amt))

    # anomaly 1: duplicate invoice numbers (same vendor + number, different file)
    for k in range(2):
        c, v, inv_no, amt = random.choice(used_numbers)
        make_invoice(c, v, inv_no, rand_date(), amt, anomaly="duplicate_invoice",
                     filename=f"invoice_{inv_no}_DUP{k + 1}.pdf")

    # anomaly 2: amounts far from the vendor's average
    for k in range(2):
        c, v = random.choice(clients), random.choice(VENDORS)
        make_invoice(c, v, f"INV-2025-{2000 + k}", rand_date(),
                     round(typical[v] * random.uniform(6, 10), 2), anomaly="unusual_amount",
                     filename=f"invoice_OUTLIER{k + 1}.pdf")

    # anomaly 3: missing fields
    for k, field in enumerate(["invoice_number", "date", "total_amount"]):
        c, v = random.choice(clients), random.choice(VENDORS)
        make_invoice(c, v, f"INV-2025-{3000 + k}", rand_date(),
                     round(typical[v] * random.uniform(0.9, 1.1), 2), missing=[field],
                     anomaly="missing_field", filename=f"invoice_MISSING_{field}.pdf")


# ---------- bank statements (FBAR) ----------
def make_statement(client, country, account_last4, max_balance_local, filename):
    currency, bank = COUNTRIES[country]
    start = rand_date(date(2025, 1, 1), 300)
    rows = [["Date", "Description", "Debit", "Credit", "Balance"]]
    balance = round(max_balance_local * random.uniform(0.4, 0.8), 2)
    peak = balance
    for i in range(12):
        d = start + timedelta(days=i * 2)
        debit = credit = 0.0
        if random.random() < 0.5:
            debit = round(random.uniform(0.01, 0.08) * max_balance_local, 2)
        else:
            credit = round(random.uniform(0.01, 0.15) * max_balance_local, 2)
        balance = round(balance - debit + credit, 2)
        peak = max(peak, balance)
        rows.append([d.isoformat(), random.choice(["Transfer", "Salary", "Card payment", "Utility"]),
                     f"{debit:,.2f}" if debit else "", f"{credit:,.2f}" if credit else "",
                     f"{balance:,.2f}"])
    story = [Paragraph(f"<b>{bank}</b>", styles["Title"]),
             Paragraph("ACCOUNT STATEMENT", styles["Heading2"]),
             table([["Account holder:", client["name"]], ["Account no:", f"XXXX-XXXX-{account_last4}"],
                    ["Country:", country], ["Currency:", currency]], [1.5 * inch, 3 * inch], header=False),
             Spacer(1, 12), table(rows, [1 * inch, 1.8 * inch, 1.1 * inch, 1.1 * inch, 1.3 * inch]),
             Spacer(1, 10), Paragraph(f"<b>Maximum balance during period: {peak:,.2f} {currency}</b>",
                                      styles["Normal"]),
             Paragraph("SYNTHETIC SAMPLE DOCUMENT - NOT REAL", styles["Italic"])]
    build(OUT / filename, story)
    usd = round(peak * TO_USD[currency], 2)
    ground_truth.append({
        "file": filename, "doc_type": "bank_statement", "client_id": client["client_id"],
        "vendor": bank, "invoice_number": "", "date": start.isoformat(), "currency": currency,
        "total_amount": peak, "anomaly": "none",
        "note": f"country={country};account_last4={account_last4};max_balance_usd={usd}",
    })
    return usd


def generate_statements(clients):
    """Each client gets 1-3 foreign accounts. Some clients cross $10k aggregate."""
    fbar_rows = []
    for c in clients:
        target_over = random.random() < 0.5  # ~half cross the FBAR threshold
        n_acc = random.randint(1, 3)
        total_usd = 0
        for a in range(n_acc):
            country = random.choice(list(COUNTRIES))
            currency = COUNTRIES[country][0]
            base_usd = random.uniform(6000, 14000) if target_over else random.uniform(500, 2800)
            max_local = base_usd / TO_USD[currency]
            last4 = f"{random.randint(0, 9999):04d}"
            total_usd += make_statement(c, country, last4, max_local,
                                        f"statement_client{c['client_id']}_acc{a + 1}.pdf")
        fbar_rows.append({"client_id": c["client_id"], "client": c["name"],
                          "aggregate_max_usd": round(total_usd, 2),
                          "fbar_required": total_usd > 10000})
    with open(OUT / "fbar_ground_truth.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fbar_rows[0].keys())
        w.writeheader()
        w.writerows(fbar_rows)


# ---------- simplified W-8BEN / 1099-NEC ----------
def make_w8ben(client, filename, missing_tin=False):
    tin = "" if missing_tin else client["tin"]
    story = [Paragraph("<b>Form W-8BEN (Simplified Synthetic Sample)</b>", styles["Title"]),
             Paragraph("Certificate of Foreign Status of Beneficial Owner - training sample only, "
                       "not the official IRS form.", styles["Italic"]), Spacer(1, 12),
             table([["1. Name of beneficial owner", client["name"]],
                    ["2. Country of citizenship", client["country"]],
                    ["3. Permanent residence country", client["country"]],
                    ["4. Foreign tax identifying number", tin],
                    ["5. Treaty claim", f"Resident of {client['country']} within tax treaty with the U.S."],
                    ["6. Signature date", rand_date().isoformat()]],
                   [2.6 * inch, 3.6 * inch], header=False),
             Spacer(1, 10), Paragraph("SAMPLE DOCUMENT - NOT REAL", styles["Italic"])]
    build(OUT / filename, story)
    ground_truth.append({
        "file": filename, "doc_type": "w8ben", "client_id": client["client_id"], "vendor": "",
        "invoice_number": "", "date": "", "currency": "", "total_amount": "",
        "anomaly": "missing_field" if missing_tin else "none",
        "note": "foreign_tin missing" if missing_tin else "",
    })


def make_1099(client, filename):
    amount = round(random.uniform(2000, 45000), 2)
    payer = random.choice(VENDORS)
    story = [Paragraph("<b>Form 1099-NEC (Simplified Synthetic Sample)</b>", styles["Title"]),
             Paragraph("Nonemployee Compensation - training sample only, not the official IRS form.",
                       styles["Italic"]), Spacer(1, 12),
             table([["Payer name", payer], ["Payer TIN", f"SYN-{random.randint(100000, 999999)}"],
                    ["Recipient name", client["name"]], ["Recipient TIN", client["tin"]],
                    ["Tax year", "2025"], ["Box 1 - Nonemployee compensation", f"USD {amount:,.2f}"]],
                   [2.6 * inch, 3.6 * inch], header=False),
             Spacer(1, 10), Paragraph("SAMPLE DOCUMENT - NOT REAL", styles["Italic"])]
    build(OUT / filename, story)
    ground_truth.append({
        "file": filename, "doc_type": "1099_nec", "client_id": client["client_id"], "vendor": payer,
        "invoice_number": "", "date": "2025-12-31", "currency": "USD", "total_amount": amount,
        "anomaly": "none", "note": "",
    })


# ---------- main ----------
if __name__ == "__main__":
    clients = make_clients(8)
    generate_invoices(clients)
    generate_statements(clients)
    for i, c in enumerate(clients):
        make_w8ben(c, f"w8ben_client{c['client_id']}.pdf", missing_tin=(i == 3))  # one planted gap
    for c in random.sample(clients, 4):
        make_1099(c, f"form1099_client{c['client_id']}.pdf")

    with open(OUT / "ground_truth.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=ground_truth[0].keys())
        w.writeheader()
        w.writerows(ground_truth)

    print(f"Done. {len(ground_truth)} PDFs in ./{OUT}/")
    print("See ground_truth.csv (extraction + anomaly answers) and fbar_ground_truth.csv (FBAR answers).")

Done. 50 PDFs in ./samples/
See ground_truth.csv (extraction + anomaly answers) and fbar_ground_truth.csv (FBAR answers).


In [2]:
pip install reportlab

  Using cached reportlab-5.0.1-py3-none-any.whl.metadata (1.6 kB)
Using cached reportlab-5.0.1-py3-none-any.whl (2.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
"""
Document generator for the Cross-Border Tax Document Assistant.

Builds one coherent set of records for a fictional company group
("Halden & Rowe") so the PDFs read like a real company's files:
  - vendor invoices billed to the group's entities
  - foreign bank statements (for FBAR aggregation per entity)
  - W-8BEN-style forms from foreign contractors
  - 1099-NEC-style forms for US contractors

The PDFs contain no labels or hints about anomalies. All answers live in
separate CSVs so you can score your pipeline:
  ground_truth.csv, fbar_ground_truth.csv, entities.csv, contractors.csv

Setup:  pip install reportlab
Run:    python generate_company_documents.py
Output: documents/  (PDFs + CSVs)

NOTE: every name, address and ID is invented by this script. Identifier
numbers are random and could coincide with real ones by chance, so keep
this data out of anything official, and describe it in your README as
generated/sample data.
"""
import csv
import random
import re
from datetime import date, timedelta
from pathlib import Path

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

random.seed(42)  # reproducible
OUT = Path("documents")
OUT.mkdir(exist_ok=True)
styles = getSampleStyleSheet()
YEAR = 2025
N_NORMAL_INVOICES = 24

# ---------------- the company group ----------------
ENTITIES = [
    {"entity_id": 1, "name": "Halden & Rowe Advisory Inc.", "address": "2100 Congress Ave, Suite 410, Austin, TX 78701", "ein": "84-3172906"},
    {"entity_id": 2, "name": "Halden & Rowe Analytics LLC", "address": "1550 Wewatta St, Floor 9, Denver, CO 80202", "ein": "86-2048113"},
    {"entity_id": 3, "name": "Halden Digital Services Inc.", "address": "301 Fayetteville St, Suite 1200, Raleigh, NC 27601", "ein": "85-6631407"},
    {"entity_id": 4, "name": "Rowe Capital Partners LLC", "address": "233 N Michigan Ave, Suite 2250, Chicago, IL 60601", "ein": "87-1925538"},
]

VENDORS = [  # name, prefix, address, currency, typical invoice amount
    ("Kestrel Cloud Services", "KCS", "48 Mill Lane, Bangalore 560001, India", "INR", 185000),
    ("Tamarind Legal LLP", "TLL", "12 Marine Drive, Mumbai 400020, India", "INR", 240000),
    ("Lumora Analytics Ltd", "LUM", "7 Farringdon Rd, London EC1M 3HN, United Kingdom", "GBP", 3200),
    ("Ostrava Freight GmbH", "OFG", "Hafenstrasse 22, 20457 Hamburg, Germany", "EUR", 2800),
    ("Pinecrest Marketing Pte Ltd", "PCM", "88 Robinson Rd, #12-01, Singapore 068898", "SGD", 4100),
    ("Vantor IT Solutions", "VIT", "3-5-1 Marunouchi, Chiyoda-ku, Tokyo 100-0005, Japan", "JPY", 520000),
    ("Brightwell Staffing Inc.", "BWS", "900 Peachtree St NE, Atlanta, GA 30309, USA", "USD", 3600),
]
CURRENCY_FORMAT = {"JPY": 0, "INR": 2, "GBP": 2, "EUR": 2, "SGD": 2, "USD": 2}

# foreign banks holding the group's overseas accounts: country -> (currency, bank)
BANKS = {
    "India": ("INR", "Crestline Bank of India"),
    "United Kingdom": ("GBP", "Northgate Bank plc"),
    "Germany": ("EUR", "Rheinland Handelsbank AG"),
    "Singapore": ("SGD", "Merlion Commercial Bank"),
    "Japan": ("JPY", "Sakuragawa Bank"),
}
TO_USD = {"INR": 0.012, "GBP": 1.27, "EUR": 1.08, "SGD": 0.74, "JPY": 0.0067, "USD": 1.0}

FIRST = ["Aarav", "Meera", "Liam", "Sofia", "Kenji", "Amara", "Lucas", "Ananya", "Noah", "Elena", "Priya", "Marcus"]
LAST = ["Sharma", "Patel", "Tanaka", "Silva", "Muller", "Okafor", "Rossi", "Nair", "Kim", "Dubois", "Reyes", "Novak"]
US_STREETS = ["Maple Ave", "Oak St", "Cedar Blvd", "Lakeview Dr", "Elm St"]
US_CITIES = [("Austin", "TX", "78704"), ("Portland", "OR", "97205"), ("Columbus", "OH", "43215"), ("Tampa", "FL", "33602")]

ground_truth = []


# ---------------- helpers ----------------
def rand_date(start=date(YEAR, 1, 1), days=330):
    return start + timedelta(days=random.randint(0, days))


def money(amount, currency):
    return f"{amount:,.{CURRENCY_FORMAT.get(currency, 2)}f}"


def slug(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def build(path, story):
    SimpleDocTemplate(str(path), pagesize=letter, topMargin=0.7 * inch, bottomMargin=0.7 * inch).build(story)


def table(data, col_widths=None, header=True):
    t = Table(data, colWidths=col_widths)
    style = [("GRID", (0, 0), (-1, -1), 0.5, colors.grey), ("FONTSIZE", (0, 0), (-1, -1), 9),
             ("VALIGN", (0, 0), (-1, -1), "TOP")]
    if header:
        style += [("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey), ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold")]
    t.setStyle(TableStyle(style))
    return t


def write_csv(name, rows):
    with open(OUT / name, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)


def foreign_tin(country):
    letters = "".join(random.choices("ABCDEFGHJKLMNPQRSTUVWXYZ", k=5))
    return {
        "India": f"{letters}{random.randint(1000, 9999)}{random.choice('ABCDEFGH')}",
        "United Kingdom": f"{random.randint(10000, 99999)} {random.randint(10000, 99999)}",
        "Germany": f"{random.randint(10, 99)}/{random.randint(100, 999)}/{random.randint(10000, 99999)}",
        "Singapore": f"S{random.randint(1000000, 9999999)}{random.choice('ABCDEFGHJK')}",
        "Japan": f"{random.randint(1000, 9999)} {random.randint(1000, 9999)} {random.randint(1000, 9999)}",
    }[country]


# ---------------- contractors ----------------
def make_contractors(n_foreign=6, n_us=4):
    rows = []
    for i in range(1, n_foreign + 1):
        country = random.choice(list(BANKS))
        rows.append({"contractor_id": i, "name": f"{random.choice(FIRST)} {random.choice(LAST)}",
                     "country": country, "us_person": False, "tin": foreign_tin(country), "address": ""})
    for j in range(n_us):
        city, st, zipc = random.choice(US_CITIES)
        rows.append({"contractor_id": n_foreign + j + 1, "name": f"{random.choice(FIRST)} {random.choice(LAST)}",
                     "country": "United States", "us_person": True,
                     "tin": f"{random.randint(900, 999)}-{random.randint(70, 88)}-{random.randint(1000, 9999)}",
                     "address": f"{random.randint(100, 9800)} {random.choice(US_STREETS)}, {city}, {st} {zipc}"})
    write_csv("contractors.csv", rows)
    return rows


# ---------------- invoices ----------------
def make_invoice(entity, vendor, inv_no, inv_date, amount, missing=None, anomaly="none", filename=None):
    name, prefix, addr, currency, _ = vendor
    missing = missing or []
    filename = filename or f"invoice_{slug(name)}_{inv_no or inv_date.isoformat()}.pdf"
    tax_rate = 0.18 if currency == "INR" else 0.0
    subtotal = round(amount / (1 + tax_rate), CURRENCY_FORMAT[currency])
    tax = round(amount - subtotal, CURRENCY_FORMAT[currency])

    story = [Paragraph(f"<b>{name}</b>", styles["Title"]), Paragraph(addr, styles["Normal"]), Spacer(1, 10),
             Paragraph("INVOICE", styles["Heading2"])]
    meta = []
    if "invoice_number" not in missing:
        meta.append(["Invoice No:", inv_no])
    if "date" not in missing:
        meta.append(["Invoice Date:", inv_date.isoformat()])
    meta += [["Due Date:", (inv_date + timedelta(days=30)).isoformat()],
             ["Terms:", "Net 30"], ["PO Number:", f"PO-{random.randint(10000, 99999)}"],
             ["Bill To:", f"{entity['name']}\n{entity['address']}"]]
    story += [table(meta, [1.5 * inch, 4 * inch], header=False), Spacer(1, 14)]

    qty = random.randint(1, 10)
    unit = round(subtotal / qty, CURRENCY_FORMAT[currency])
    story.append(table([["Description", "Qty", f"Unit Price ({currency})", f"Amount ({currency})"],
                        [random.choice(["Professional services", "Consulting - monthly retainer", "Project deliverables"]),
                         qty, money(unit, currency), money(unit * qty, currency)]],
                       [3 * inch, 0.7 * inch, 1.5 * inch, 1.4 * inch]))
    story.append(Spacer(1, 10))
    if "total_amount" not in missing:
        lines = [["Subtotal:", money(subtotal, currency)]]
        if tax:
            lines.append(["GST (18%):", money(tax, currency)])
        lines.append([f"Total Due ({currency}):", money(amount, currency)])
        story.append(table(lines, [2 * inch, 1.6 * inch], header=False))
    story += [Spacer(1, 12), Paragraph("Please remit payment by wire transfer. Thank you for your business.", styles["Italic"])]
    build(OUT / filename, story)

    ground_truth.append({
        "file": filename, "doc_type": "invoice", "entity_id": entity["entity_id"], "counterparty": name,
        "invoice_number": "" if "invoice_number" in missing else inv_no,
        "date": "" if "date" in missing else inv_date.isoformat(), "currency": currency,
        "total_amount": "" if "total_amount" in missing else amount, "anomaly": anomaly,
        "note": ",".join(missing),
    })


def generate_invoices():
    seq = {v[1]: 100 for v in VENDORS}
    issued = []
    for _ in range(N_NORMAL_INVOICES):
        e, v = random.choice(ENTITIES), random.choice(VENDORS)
        seq[v[1]] += random.randint(1, 4)
        inv_no = f"{v[1]}-{YEAR}-{seq[v[1]]:04d}"
        amt = round(v[4] * random.uniform(0.85, 1.15), CURRENCY_FORMAT[v[3]])
        d = rand_date()
        make_invoice(e, v, inv_no, d, amt)
        issued.append((e, v, inv_no, amt))

    for k in range(2):  # duplicate invoice numbers (same vendor + number, resent under a new file name)
        e, v, inv_no, amt = random.choice(issued)
        make_invoice(e, v, inv_no, rand_date(), amt, anomaly="duplicate_invoice",
                     filename=f"invoice_{slug(v[0])}_{inv_no}_resend{k + 1}.pdf")

    for _ in range(2):  # amounts far above the vendor's normal range
        e, v = random.choice(ENTITIES), random.choice(VENDORS)
        seq[v[1]] += 1
        make_invoice(e, v, f"{v[1]}-{YEAR}-{seq[v[1]]:04d}", rand_date(),
                     round(v[4] * random.uniform(6, 10), CURRENCY_FORMAT[v[3]]), anomaly="unusual_amount")

    for field in ["invoice_number", "date", "total_amount"]:  # incomplete invoices
        e, v = random.choice(ENTITIES), random.choice(VENDORS)
        seq[v[1]] += 1
        make_invoice(e, v, f"{v[1]}-{YEAR}-{seq[v[1]]:04d}", rand_date(),
                     round(v[4] * random.uniform(0.9, 1.1), CURRENCY_FORMAT[v[3]]),
                     missing=[field], anomaly="missing_field")


# ---------------- bank statements (FBAR) ----------------
def make_statement(entity, country, last4, max_local, filename):
    currency, bank = BANKS[country]
    start = rand_date(date(YEAR, 1, 1), 300)
    opening = round(max_local * random.uniform(0.4, 0.8), 2)
    balance, peak = opening, opening
    rows = [["Date", "Description", "Debit", "Credit", "Balance"]]
    for i in range(12):
        d = start + timedelta(days=i * 2)
        debit = credit = 0.0
        if random.random() < 0.5:
            debit = round(random.uniform(0.01, 0.08) * max_local, 2)
        else:
            credit = round(random.uniform(0.01, 0.15) * max_local, 2)
        balance = round(balance - debit + credit, 2)
        peak = max(peak, balance)
        rows.append([d.isoformat(), random.choice(["Wire transfer", "Client receipt", "Card payment", "Utilities", "Payroll"]),
                     money(debit, currency) if debit else "", money(credit, currency) if credit else "", money(balance, currency)])
    story = [Paragraph(f"<b>{bank}</b>", styles["Title"]), Paragraph("ACCOUNT STATEMENT", styles["Heading2"]),
             table([["Account holder:", entity["name"]], ["Account no:", f"****{last4}"],
                    ["Branch country:", country], ["Currency:", currency],
                    ["Statement period:", f"{start.isoformat()} to {(start + timedelta(days=24)).isoformat()}"],
                    ["Opening balance:", money(opening, currency)]], [1.6 * inch, 3.4 * inch], header=False),
             Spacer(1, 12), table(rows, [1 * inch, 1.8 * inch, 1.1 * inch, 1.1 * inch, 1.3 * inch]), Spacer(1, 10),
             Paragraph(f"<b>Closing balance: {money(balance, currency)} {currency}</b>", styles["Normal"]),
             Paragraph(f"Highest balance in period: {money(peak, currency)} {currency}", styles["Normal"])]
    build(OUT / filename, story)
    usd = round(peak * TO_USD[currency], 2)
    ground_truth.append({
        "file": filename, "doc_type": "bank_statement", "entity_id": entity["entity_id"], "counterparty": bank,
        "invoice_number": "", "date": start.isoformat(), "currency": currency, "total_amount": peak, "anomaly": "none",
        "note": f"country={country};account_last4={last4};max_balance_usd={usd}",
    })
    return usd


def generate_statements():
    fbar = []
    for i, e in enumerate(ENTITIES):
        over = i % 2 == 0  # two entities cross the $10,000 aggregate threshold, two do not
        total = 0
        for a in range(random.randint(1, 3)):
            country = random.choice(list(BANKS))
            base_usd = random.uniform(6000, 14000) if over else random.uniform(500, 2800)
            total += make_statement(e, country, f"{random.randint(0, 9999):04d}", base_usd / TO_USD[BANKS[country][0]],
                                    f"statement_{slug(e['name'])}_acct{a + 1}.pdf")
        fbar.append({"entity_id": e["entity_id"], "entity": e["name"], "aggregate_max_usd": round(total, 2),
                     "fbar_required": total > 10000})
    write_csv("fbar_ground_truth.csv", fbar)


# ---------------- contractor forms ----------------
def make_w8ben(c, filename, missing_tin=False):
    tin = "" if missing_tin else c["tin"]
    story = [Paragraph("<b>Form W-8BEN</b>", styles["Title"]),
             Paragraph("Certificate of Foreign Status of Beneficial Owner for United States Tax Withholding (Individuals)",
                       styles["Normal"]), Spacer(1, 12),
             table([["1. Name of individual who is the beneficial owner", c["name"]],
                    ["2. Country of citizenship", c["country"]],
                    ["3. Permanent residence country", c["country"]],
                    ["4. Foreign tax identifying number", tin],
                    ["5. Treaty benefits claimed", f"Resident of {c['country']} within the meaning of the income tax treaty with the United States"],
                    ["6. Date signed", rand_date().isoformat()]], [2.8 * inch, 3.4 * inch], header=False)]
    build(OUT / filename, story)
    ground_truth.append({
        "file": filename, "doc_type": "w8ben", "entity_id": "", "counterparty": c["name"], "invoice_number": "",
        "date": "", "currency": "", "total_amount": "", "anomaly": "missing_field" if missing_tin else "none",
        "note": "foreign_tin missing" if missing_tin else f"contractor_id={c['contractor_id']}",
    })


def make_1099(c, filename):
    payer = random.choice(ENTITIES)
    amount = round(random.uniform(2000, 45000), 2)
    story = [Paragraph("<b>Form 1099-NEC</b>", styles["Title"]), Paragraph("Nonemployee Compensation - Tax Year 2025", styles["Normal"]),
             Spacer(1, 12),
             table([["Payer name", payer["name"]], ["Payer address", payer["address"]], ["Payer TIN", payer["ein"]],
                    ["Recipient name", c["name"]], ["Recipient address", c["address"]], ["Recipient TIN", c["tin"]],
                    ["Box 1 - Nonemployee compensation", f"$ {amount:,.2f}"]], [2.6 * inch, 3.6 * inch], header=False)]
    build(OUT / filename, story)
    ground_truth.append({
        "file": filename, "doc_type": "1099_nec", "entity_id": payer["entity_id"], "counterparty": c["name"],
        "invoice_number": "", "date": f"{YEAR}-12-31", "currency": "USD", "total_amount": amount, "anomaly": "none",
        "note": f"contractor_id={c['contractor_id']}",
    })


# ---------------- main ----------------
if __name__ == "__main__":
    write_csv("entities.csv", ENTITIES)
    contractors = make_contractors()
    generate_invoices()
    generate_statements()
    foreign = [c for c in contractors if not c["us_person"]]
    for i, c in enumerate(foreign):
        make_w8ben(c, f"w8ben_{slug(c['name'])}.pdf", missing_tin=(i == 3))  # one form with a blank TIN
    for c in [c for c in contractors if c["us_person"]]:
        make_1099(c, f"form1099nec_{slug(c['name'])}.pdf")
    write_csv("ground_truth.csv", ground_truth)
    print(f"Done. {len(ground_truth)} PDFs in ./{OUT}/ plus ground_truth.csv, fbar_ground_truth.csv, entities.csv, contractors.csv")

Done. 51 PDFs in ./documents/ plus ground_truth.csv, fbar_ground_truth.csv, entities.csv, contractors.csv
